---
title: "Exercise 2. Prepare MAGMA Input Files"
subtitle: "From cleaned SNP-level GWAS output to the two tabular inputs MAGMA needs"
format:
  html:
    toc: true
    toc-depth: 3
    number-sections: true
---

# Overview
MAGMA does not read the full GWAS summary-statistics table directly in the format used in notebook 1. Instead, we extract two simpler files: one containing SNP positions and one containing p-values. This notebook creates both a full dataset and a smaller classroom-friendly demo subset.

::: {.callout-note}
## Why this notebook matters
The remaining MAGMA notebooks assume that `input/magma/` is created here. If you skip this step, the annotation and gene-analysis notebooks will not have the required SNP location and p-value files.
:::

# Learning goals and prompts

After this notebook you should be able to explain:
- why MAGMA needs SNP location and p-value files separately
- why teaching workflows often include a reduced demo dataset
- how QC choices made earlier propagate into downstream enrichment results

:::{.callout-note}
**Questions**
1. Why might a top-hit demo set be useful for teaching, even if it is not ideal for inference?
2. Why do we restrict to chromosomes 1 to 22 here?
3. What problems would appear later if SNP identifiers were duplicated or inconsistent across files?
:::

In [1]:
library(vroom)
library(dplyr)

sumstats_path <- "input/ADHD2022_iPSYCH_deCODE_PGC.meta.gz"
magma_input_dir <- "input/magma"
magma_output_dir <- "output/magma"
dir.create(magma_output_dir, recursive = TRUE, showWarnings = FALSE)


Attaching package: ‘dplyr’


The following objects are masked from ‘package:stats’:

    filter, lag


The following objects are masked from ‘package:base’:

    intersect, setdiff, setequal, union




# Reload the GWAS summary statistics

This notebook repeats the same basic filtering logic as notebook 1 so that the MAGMA inputs are aligned with the exploratory QC decisions. In a production pipeline, you might centralise this logic to avoid accidental drift between steps.

In [2]:
sumstats <- vroom(sumstats_path, show_col_types = FALSE) %>%
  mutate(MAF = pmin(FRQ_A_38691, 1 - FRQ_A_38691)) %>%
  filter(INFO >= 0.8, MAF >= 0.01)

cat(sprintf("QC-passing SNPs: %s\n", format(nrow(sumstats), big.mark = ",")))

sumstats %>%
  summarise(
    min_p = min(P, na.rm = TRUE),
    max_p = max(P, na.rm = TRUE),
    min_chr = min(CHR, na.rm = TRUE),
    max_chr = max(CHR, na.rm = TRUE)
  )

QC-passing SNPs: 6,774,125


min_p,max_p,min_chr,max_chr
<dbl>,<dbl>,<dbl>,<dbl>
9.033e-15,1,1,22


# Full MAGMA input files

The full version keeps all QC-passing autosomal SNPs. This is the input you would use for a serious analysis. The file split is simple but important:
- `snp_loc_full.txt`: SNP, chromosome, and position
- `pval_full.txt`: SNP and p-value

::: {.callout-important}
MAGMA expects the same SNP identifiers to line up across these files. If one file contains SNPs missing from the other, you will either lose data or produce confusing downstream failures.
:::

In [5]:
snp_loc_full <- sumstats %>%
  filter(CHR %in% 1:22) %>%
  select(SNP, CHR, BP)

pval_full <- sumstats %>%
  filter(CHR %in% 1:22) %>%
  select(SNP, P)

vroom_write(snp_loc_full, file.path(magma_output_dir, "snp_loc_full.txt"), delim = "\t", col_names = TRUE)
vroom_write(pval_full, file.path(magma_output_dir, "pval_full.txt"), delim = "\t", col_names = TRUE)

tibble(file = c("snp_loc_full.txt", "pval_full.txt"), rows = c(nrow(snp_loc_full), nrow(pval_full)))

file,rows
<chr>,<int>
snp_loc_full.txt,6774125
pval_full.txt,6774125


# Demo MAGMA input files

For teaching, we often prefer something that runs in minutes instead of tens of minutes. Here we keep only SNPs with `P < 5e-7`, which is not a substitute for the full analysis but is enough to demonstrate the pipeline live.

**Question for students:** how might a demo subset distort gene-level or tissue-level conclusions relative to the full analysis?

In [6]:
DEMO_THRESH <- 5e-7
demo_snps <- sumstats %>%
  filter(CHR %in% 1:22, P < DEMO_THRESH)

snp_loc_demo <- demo_snps %>% select(SNP, CHR, BP)
pval_demo <- demo_snps %>% select(SNP, P)

vroom_write(snp_loc_demo, file.path(magma_output_dir, "snp_loc_demo.txt"), delim = "\t", col_names = TRUE)
vroom_write(pval_demo, file.path(magma_output_dir, "pval_demo.txt"), delim = "\t", col_names = TRUE)

demo_snps %>%
  count(CHR, name = "n_SNPs") %>%
  arrange(CHR)

CHR,n_SNPs
<dbl>,<int>
1,422
2,52
3,244
4,124
5,382
6,25
7,87
8,66
9,8


In [7]:
cat("Preview of the full SNP location file:\n")
head(snp_loc_full)

cat("\nPreview of the full p-value file:\n")
head(pval_full)

Preview of the full SNP location file:


SNP,CHR,BP
<chr>,<dbl>,<dbl>
rs62513865,8,101592213
rs79643588,8,106973048
rs17396518,8,108690829
rs983166,8,108681675
rs28842593,8,103044620
rs7014597,8,104152280



Preview of the full p-value file:


SNP,P
<chr>,<dbl>
rs62513865,0.8325
rs79643588,0.7967
rs17396518,0.6876
rs983166,0.5956
rs28842593,0.2081
rs7014597,0.9679


# Wrap-up

You should now have four files in `output/magma/`: full and demo versions of the SNP-location and p-value tables. The next notebook uses the SNP-location files together with a gene-location reference to assign SNPs to genes.

::: {.callout-tip}
## Reflection prompts
1. What assumptions are baked into the decision to keep only autosomes here?
2. If your GWAS used a different genome build, which of the next MAGMA inputs would become incompatible?
3. Why is this step mostly about formatting and harmonisation rather than statistics?
:::